# Reporte 02: Nivel de Cobertura Espacial Capilar (C)

## 1. Justificación Teórica
El presente indicador evalúa la equidad espacial de la red de transporte masivo. Se define un isócrono de accesibilidad peatonal (radio de caminata o *Buffer*) alrededor de cada estación, típicamente de **800 metros** (equivalente a 10-15 minutos caminando).  
**Definir área metropolitana en un polígono de CDXM y algunos municipios de EDOMEX**  


El Nivel de Cobertura ($C$) se calcula intersectando el área de influencia de las estaciones ($A_{cubierta}$) contra la superficie geodésica total de la demarcación política ($A_{total}$):

$$C = \left( \frac{A_{cubierta}}{A_{total}} \right) \times 100$$  

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import warnings
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Silenciar advertencias futuras para un reporte limpio
warnings.filterwarnings('ignore', category=FutureWarning)

proyecto_path = os.path.abspath('..')
if proyecto_path not in sys.path:
    sys.path.append(proyecto_path)

from src.infrastructure.go_client.client import fetch_full_network
from src.infrastructure.go_client.client_spatial import fetch_territorial_polygons
from src.core.algorithms.topologicalIndicators.spatial_coverate import SpatialCoverageAnalyzer

print("Extrayendo datos espaciales y de red desde Go...")
transporte_data = await fetch_full_network()
poligonos_data = await fetch_territorial_polygons(entidades=["Ciudad de México", "México"])

analizador_cobertura = SpatialCoverageAnalyzer(transporte_data, poligonos_data)
print("✅ Datos cargados y proyectados a métrica local (EPSG:32614).")

# Lista estricta para segmentación espacial
ALCALDIAS_CDMX = [
    'Álvaro Obregón', 'Azcapotzalco', 'Benito Juárez', 'Coyoacán', 
    'Cuajimalpa de Morelos', 'Cuauhtémoc', 'Gustavo A. Madero', 'Iztacalco', 
    'Iztapalapa', 'La Magdalena Contreras', 'Miguel Hidalgo', 'Milpa Alta', 
    'Tláhuac', 'Tlalpan', 'Venustiano Carranza', 'Xochimilco'
]

Extrayendo datos espaciales y de red desde Go...
2026-05-30 22:20:59 | INFO     | VFT_Model | Construyendo el puente hacia el módulo espacial de Go...
2026-05-30 22:20:59 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonEstacion
2026-05-30 22:20:59 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonLinea
2026-05-30 22:21:00 | INFO     | VFT_Model | Red extraída: 22766 entidades espaciales listas para VFT.
2026-05-30 22:21:00 | INFO     | VFT_Model | Solicitando polígonos territoriales a APIMETRO para: ['Ciudad de México', 'México']
2026-05-30 22:21:00 | INFO     | VFT_Model | Polígonos combinados exitosamente. Total features: 157
✅ Datos cargados y proyectados a métrica local (EPSG:32614).


In [3]:
RADIO_METROS = 800.0

print(f"Calculando cobertura general agregada (Isócrono: {RADIO_METROS}m)...")
df_general = analizador_cobertura.calculate_general_coverage(radio_caminable_m=RADIO_METROS)

print("Calculando coberturas independientes por sistema...")
dict_sistemas = analizador_cobertura.calculate_coverage_by_system(radio_caminable_m=RADIO_METROS)

print("✅ Cálculos finalizados.")

Calculando cobertura general agregada (Isócrono: 800.0m)...
Calculando coberturas independientes por sistema...
✅ Cálculos finalizados.


In [ ]:
display(Markdown(f"## 1. Escenario Base: Cobertura Total del Sistema de Transporte"))
display(Markdown("Análisis de las demarcaciones con **mayor rezago** (menor porcentaje de cobertura espacial), evidenciando las zonas de exclusión al transporte masivo."))

# Separar CDMX y EDOMEX
df_general_cdmx = df_general[df_general['Demarcacion'].isin(ALCALDIAS_CDMX)]
df_general_edomex = df_general[~df_general['Demarcacion'].isin(ALCALDIAS_CDMX)]

# Extraer los peores 5 (Bottom 5)
peores_cdmx = df_general_cdmx.sort_values('Cobertura_Porcentaje', ascending=True).head(5)
peores_edomex = df_general_edomex.sort_values('Cobertura_Porcentaje', ascending=True).head(5)

_cols_tab = ['Demarcacion', 'Area_Total_km2', 'Area_Cubierta_km2', 'Cobertura_Porcentaje']

display(Markdown(f"### 🔻 Peores 5 Coberturas - Alcaldías (CDMX)"))
display(peores_cdmx[_cols_tab])

display(Markdown(f"### 🔻 Peores 5 Coberturas - Municipios (EDOMEX)"))
display(peores_edomex[_cols_tab])

# Preparar y mostrar el Mapa General
gdf_mapa_general = analizador_cobertura.gdf_poligonos.copy()
gdf_mapa_general = gdf_mapa_general.merge(df_general, left_on='nombre', right_on='Demarcacion', how='left')

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_title(f"Mapa de Calor: Cobertura Total Territorial (Radio: {RADIO_METROS}m)", fontsize=14, fontweight='bold')

gdf_mapa_general.plot(
    column='Cobertura_Porcentaje', cmap='YlGnBu', linewidth=0.8, ax=ax, edgecolor='0.5',
    legend=True, legend_kwds={'label': "Cobertura (%)", 'orientation': "horizontal", 'shrink': 0.6},
    missing_kwds={"color": "lightgrey", "label": "Sin Datos"}
)

# Plotear la mancha unida
buffers_gen = analizador_cobertura.gdf_estaciones.geometry.buffer(RADIO_METROS)
mancha_gen = gpd.GeoSeries([buffers_gen.unary_union])
mancha_gen.plot(ax=ax, color='red', alpha=0.3, linewidth=0)

ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown(f"## 2. Desglose Operativo por Sistema"))
display(Markdown("A continuación se evalúa el nivel de penetración territorial de cada sistema de manera aislada, destacando las demarcaciones centrales de la Ciudad de México con mayor nivel de servicio."))

# Filtrar sistemas válidos
sistemas_plot = [sis for sis in dict_sistemas.keys() if not dict_sistemas[sis].empty]

for sistema in sistemas_plot:
    df_sis = dict_sistemas[sistema]
    
    # Filtrar SOLO CDMX para el Top 5
    df_sis_cdmx = df_sis[df_sis['Demarcacion'].isin(ALCALDIAS_CDMX)]
    top5_cdmx = df_sis_cdmx.sort_values('Cobertura_Porcentaje', ascending=False).head(5)
    
    # 1. Imprimir Títulos y Tabla
    display(Markdown(f"---"))
    display(Markdown(f"### 🚇 Sistema: {str(sistema).upper()}"))
    display(Markdown(f"**Top 5 Alcaldías (CDMX) con mayor cobertura**"))
    _cols_sis = ['Demarcacion', 'Area_Total_km2', 'Area_Cubierta_km2', 'Cobertura_Porcentaje']
    display(top5_cdmx[_cols_sis])
    
    # 2. Dibujar su mapa
    gdf_sis = analizador_cobertura.gdf_poligonos.copy()
    gdf_sis = gdf_sis.merge(df_sis, left_on='nombre', right_on='Demarcacion', how='left')
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Mapa coroplético (escala fija 0-100 para comparar peras con peras)
    gdf_sis.plot(
        column='Cobertura_Porcentaje', cmap='YlGnBu', linewidth=0.5, ax=ax, edgecolor='0.8',
        legend=True, vmin=0, vmax=100, 
        legend_kwds={'shrink': 0.5, 'label': f'Cobertura {str(sistema).upper()} (%)'}, 
        missing_kwds={"color": "lightgrey"}
    )
    
    # Agregar las estaciones de ese sistema en específico
    estaciones_sis = analizador_cobertura.gdf_estaciones[analizador_cobertura.gdf_estaciones['sistema'] == sistema]
    if not estaciones_sis.empty:
        buffers_sis = estaciones_sis.geometry.buffer(RADIO_METROS)
        mancha_sis = gpd.GeoSeries([buffers_sis.unary_union])
        mancha_sis.plot(ax=ax, color='red', alpha=0.4, linewidth=0)
        
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [7]:
from pathlib import Path

EXPORT_PATH = Path("../../Transport-gis-zmvm-mjg/data/processed/Cobertura800m.gpkg")

# 1. Mancha de cobertura total
buffers = analizador_cobertura.gdf_estaciones.geometry.buffer(RADIO_METROS)
mancha = gpd.GeoDataFrame(geometry=[buffers.unary_union], crs="EPSG:32614")

# 2. Zona sin cobertura
zmvm = analizador_cobertura.gdf_poligonos.dissolve()
sin_cobertura_geom = zmvm.geometry.iloc[0].difference(mancha.geometry[0])
gdf_sin_cobertura = gpd.GeoDataFrame(geometry=[sin_cobertura_geom], crs="EPSG:32614")

# 3. Estaciones — castear StringDtype → object para que fiona lo acepte
estaciones = analizador_cobertura.gdf_estaciones[['sistema', 'geometry']].copy()
estaciones['sistema'] = estaciones['sistema'].astype(object)

# Exportar
mancha.to_file(EXPORT_PATH, layer="cobertura_800m", driver="GPKG")
gdf_sin_cobertura.to_file(EXPORT_PATH, layer="sin_cobertura", driver="GPKG")
estaciones.to_file(EXPORT_PATH, layer="estaciones", driver="GPKG")

print(f"✅ Exportado: {EXPORT_PATH}")

✅ Exportado: ../../Transport-gis-zmvm-mjg/data/processed/Cobertura800m.gpkg


In [10]:
from pathlib import Path

EXPORT_PATH = Path("../../Transport-gis-zmvm-mjg/data/processed/Cobertura800m.gpkg")

gdf_coro = analizador_cobertura.gdf_poligonos.copy()
gdf_coro = gdf_coro.merge(df_general, left_on='nombre', right_on='Demarcacion', how='left')
gdf_coro = gdf_coro[['nombre', 'Cobertura_Porcentaje', 'Area_Total_km2', 'Area_Cubierta_km2', 'geometry']]

gdf_coro['nombre'] = gdf_coro['nombre'].astype(object)

def clasificar_cobertura(pct):
    if   pd.isna(pct):  return "Sin datos"
    elif pct >= 60:     return "Alta >=60%"
    elif pct >= 30:     return "Media 30-60%"
    elif pct >= 10:     return "Baja 10-30%"
    else:               return "Muy baja <10%"

gdf_coro['Clasificacion'] = gdf_coro['Cobertura_Porcentaje'].apply(clasificar_cobertura).astype(object)

# engine="pyogrio" agrega la capa sin tocar las existentes
gdf_coro.to_file(EXPORT_PATH, layer="cobertura_por_alcaldia", driver="GPKG", engine="pyogrio")

print(f"✅ Exportado: {EXPORT_PATH}")
print(f"   Demarcaciones: {len(gdf_coro)}")
print(f"   Rango: {gdf_coro['Cobertura_Porcentaje'].min():.1f}% – {gdf_coro['Cobertura_Porcentaje'].max():.1f}%")
print(f"\n{'Demarcación':<35} {'Cobertura':>9}")
print("─" * 46)
for _, row in gdf_coro.nsmallest(8, 'Cobertura_Porcentaje').iterrows():
    print(f"{row['nombre']:<35} {row['Cobertura_Porcentaje']:>8.1f}%")

✅ Exportado: ../../Transport-gis-zmvm-mjg/data/processed/Cobertura800m.gpkg
   Demarcaciones: 141
   Rango: 0.0% – 100.0%

Demarcación                         Cobertura
──────────────────────────────────────────────
Acambay de Ruiz Castañeda                0.0%
Acolman                                  0.0%
Aculco                                   0.0%
Almoloya de Alquisiras                   0.0%
Almoloya de Juárez                       0.0%
Almoloya del Río                         0.0%
Amanalco                                 0.0%
Amatepec                                 0.0%
